# 03 · QAT vs PTQ on the same model
**Runtime:** one T4 GPU. **Writes:** `results_03_qat.jsonl`. **Rough time:** ~5 h (three training runs of ~90 min).

One quantizer for every row here (symmetric INT-n, group 128, round to nearest), so only training changes.
Fake quant uses a straight-through estimator (`add_fake_quant` in `edgelayer_bench.py`).

| config_id | What |
|---|---|
| `ptq_rtn_w{b}` | No training, round to nearest |
| `ft_fp16`, `ft_then_rtn_w{b}` | **Control:** the same training without fake quant, then round |
| `qat_w{b}g128` | QAT, then round the trained weights. Logs the recovery curve and % of weights that changed tick |

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
!pip install -q -U llmcompressor==0.15.0 lm-eval==0.4.13 datasets accelerate bitsandbytes   # pip warnings about gradio/cudf/bigframes are safe to ignore

## Setup
Set `PLATFORM` (`"colab"` or `"kaggle"`). The helper `edgelayer_bench.py` is fetched from the repo
(or used directly if you cloned the repo). Every finished config is saved to `RESULTS_FILE` right away,
so after a disconnect you just re-run all cells: finished configs are skipped.

In [ ]:
PLATFORM = "kaggle"                      # "colab" | "kaggle"
RESULTS_FILE = "results_03_qat.jsonl"
REPO_RAW = "https://raw.githubusercontent.com/<your-username>/edgelayer-quantization/main"   # <- replace once

import os, sys, json, urllib.request
BENCH_DIR = {"colab": "/content/drive/MyDrive/edgelayer-bench",
             "kaggle": "/kaggle/working/edgelayer-bench"}[PLATFORM]
if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")
os.makedirs(BENCH_DIR, exist_ok=True)

# Use the repo's helper if this notebook runs from a clone, else download it once
for d in ["..", "."]:
    if os.path.exists(f"{d}/edgelayer_bench.py"):
        sys.path.insert(0, os.path.abspath(d)); break
else:
    urllib.request.urlretrieve(f"{REPO_RAW}/edgelayer_bench.py", f"{BENCH_DIR}/edgelayer_bench.py")
    sys.path.insert(0, BENCH_DIR)

In [ ]:
import torch
import edgelayer_bench as eb
rd = eb.setup(BENCH_DIR, platform=PLATFORM, results_file=RESULTS_FILE)
MODEL_ID = eb.CFG["model_id"]
eb.ensure_meta()                            # parameter counts, used for bits/weight

def done_ids():
    """config_ids that already have a row with this field (default: any row)."""
    p = rd / RESULTS_FILE
    return {json.loads(l)["config_id"] for l in open(p) if l.strip()} if p.exists() else set()

print(MODEL_ID, "|", eb.ram_gb()); eb.env_info()

In [ ]:
from transformers import AutoModelForCausalLM
BITS, GROUP = [4, 3], 128
STEPS, SEQ, MICRO_BS, ACCUM, LR = 600, 1024, 2, 4, 1e-5      # 600 x 8 x 1,024 = ~4.9M tokens per run
CURVE_EVERY, CURVE_WINDOWS = 100, 10
eb.CFG["lm_eval_batch"] = "auto"

ref, tok = eb.load_fp16()
train_chunks = eb.packed_chunks(tok, "c4", n=STEPS * MICRO_BS * ACCUM, seqlen=SEQ, seed=1)

log = lambda cfg_id, **kw: eb.log_result(cfg_id, post=6, **kw)
ev = lambda model: eb.evaluate(model, tok, ref=ref)
def size_bpw(model, bits):
    s = eb.analytic_size_gb(model, bits + 16 / GROUP); return dict(size_gb=s, bpw=eb.bpw(s))
quick_ppl = lambda model: eb.perplexity(model, eb.wikitext_ids(tok), max_windows=CURVE_WINDOWS)
load32 = lambda: AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32, device_map="cuda")

## PTQ with the same quantizer

In [ ]:
for b in BITS:
    if f"ptq_rtn_w{b}" in done_ids(): continue
    m, _ = eb.load_fp16()
    eb.apply_weight_qdq(m, eb.intn_group_qdq, bits=b, group=GROUP)
    log(f"ptq_rtn_w{b}", method="RTN (same quantizer)", bits=b, group=GROUP, **size_bpw(m, b), **ev(m))
    del m; eb.free_gpu()

## Control: train without fake quant, then round

In [ ]:
if not {"ft_fp16", "ft_then_rtn_w4", "ft_then_rtn_w3"} <= done_ids():
    m = load32()
    tr = eb.train_lm(m, train_chunks, STEPS, lr=LR, micro_bs=MICRO_BS, accum=ACCUM)
    m.half()
    log("ft_fp16", method="Fine-tune only (FP16)", bits=16, steps=STEPS, train_time_s=tr["train_time_s"],
        train_peak_vram_gb=tr["train_peak_vram_gb"], **ev(m))
    latent = {n: mod.weight.detach().clone() for n, mod in eb.linear_layers(m)}
    for b in BITS:
        eb.apply_weight_qdq(m, eb.intn_group_qdq, bits=b, group=GROUP)
        log(f"ft_then_rtn_w{b}", method="Fine-tune → RTN (control)", bits=b, group=GROUP, **size_bpw(m, b), **ev(m))
        with torch.no_grad():
            for n, mod in eb.linear_layers(m): mod.weight.copy_(latent[n])
    del m, latent; eb.free_gpu()

## QAT: fake quant on during training, then round the trained weights

In [ ]:
m0 = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)          # CPU copy for tick counting
base_codes = {b: {n: eb.intn_codes(mod.weight, b, GROUP) for n, mod in eb.linear_layers(m0)} for b in BITS}
del m0

for b in BITS:
    if f"qat_w{b}g128" in done_ids(): continue
    m = load32()
    eb.add_fake_quant(m, bits=b, group=GROUP)
    tr = eb.train_lm(m, train_chunks, STEPS, lr=LR, micro_bs=MICRO_BS, accum=ACCUM,
                     eval_fn=quick_ppl, eval_every=CURVE_EVERY)
    eb.remove_fake_quant(m)
    changed = total = 0
    for n, mod in eb.linear_layers(m):
        c = eb.intn_codes(mod.weight.detach().float().cpu(), b, GROUP)
        changed += (c != base_codes[b][n]).sum().item(); total += c.numel()
    eb.apply_weight_qdq(m, eb.intn_group_qdq, bits=b, group=GROUP)
    m.half()
    log(f"qat_w{b}g128", method="QAT → RTN", bits=b, group=GROUP, steps=STEPS, lr=LR,
        train_time_s=tr["train_time_s"], train_peak_vram_gb=tr["train_peak_vram_gb"],
        pct_weights_changed_tick=round(100 * changed / total, 2), eval_curve=tr["eval_curve"],
        **size_bpw(m, b), **ev(m))
    del m; eb.free_gpu()

## Copy the results
Downloads from Kaggle can fail: the cell below prints the file so you can copy it.

In [ ]:
print(open(rd / "results_03_qat.jsonl").read())